# 03 synthetic crystal data

**Owner:** Daniel. Original notebook cells: 65, 72.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# Synthetic Crystal-Style MVP Dataset Generator
# Deterministic, non-PHI records aligned with Thomas's gateway schemas
# and Denis's booking-time no-show model features.

import os
import json
import random
from datetime import date, timedelta
import pandas as pd

SEED = 42
random.seed(SEED)

TEAM_VOLUME = "/Volumes/workspace/default/eye_style_team2"
SYNTHETIC_DATA_DIR = f"{TEAM_VOLUME}/synthetic"
CLEAN_NO_SHOW_CSV = f"{TEAM_VOLUME}/clean/cleaned_appointments_features.csv"
os.makedirs(SYNTHETIC_DATA_DIR, exist_ok=True)

# Reference "today" for demo consistency (can be adjusted to match demo date)
REFERENCE_DATE = date(2026, 11, 24)

# Pull valid Neighbourhood values from the cleaned Kaggle dataset so Denis's
# OneHotEncoder receives known prototype categories; fallback if file isn't read.
try:
    _clean_ref = pd.read_csv(CLEAN_NO_SHOW_CSV, usecols=["Neighbourhood"])
    VALID_NEIGHBOURHOODS = (
        _clean_ref["Neighbourhood"].value_counts().head(15).index.tolist()
    )
except Exception:
    VALID_NEIGHBOURHOODS = [
        "JARDIM CAMBURI", "MARIA ORTIZ", "RESISTÊNCIA",
        "JARDIM DA PENHA", "ITARARÉ", "CENTRO", "TABUAZEIRO",
        "SANTA MARTHA", "JESUS DE NAZARETH", "BONFIM"
    ]

# Fictional non-PHI patient names
FIRST_NAMES = [
    "Morgan", "Avery", "Jordan", "Taylor", "Casey", "Riley", "Quinn", "Skyler",
    "Cameron", "Drew", "Reese", "Parker", "Logan", "Peyton", "Rowan", "Sage",
    "Blake", "Hayden", "Emerson", "Finley", "Dakota", "Kendall", "Charlie",
    "Elliot", "Harper", "Sawyer", "Jules", "Devon", "Marley", "Tatum"
]
LAST_NAMES = [
    "Demo", "Sample", "Teston", "Mockley", "Vance", "Mercer", "Hale", "Brooks",
    "Sutton", "Hayes", "Ellis", "Monroe", "Price", "Bennett", "Reed", "Ward",
    "Foster", "Gray", "Cole", "West", "Shaw", "Stone", "Grant", "Bryant",
    "Hayes", "Palmer", "Knight", "Cross", "Lane", "Pierce"
]

# ---------------------------------------------------------------------
# 1. Synthetic Patients (Crystal-style: GetPatientById)
# ---------------------------------------------------------------------
patients = []
for idx in range(30):
    pid = 1001 + idx
    patients.append({
        "patientAccountId": pid,
        "firstName": FIRST_NAMES[idx],
        "lastName": LAST_NAMES[idx],
    })

# ---------------------------------------------------------------------
# 2. Synthetic Appointments (Crystal-style: GetAppointments)
#    + Companion Booking Features (Synthetic Application-Only for Denis's MLP)
# ---------------------------------------------------------------------
APPT_TIMES = [
    "08:30", "09:00", "09:30", "10:00", "10:30",
    "11:00", "13:00", "13:30", "14:00", "14:30", "15:30", "16:00"
]
APPT_TYPES = [
    "Comprehensive Eye Exam", "Contact Lens Fitting",
    "Medical Eye Follow-Up", "Frame Styling & Dispense"
]

appointments = []
scheduling_booking_features = []

for i in range(40):
    schedule_id = 5001 + i
    patient = patients[i % len(patients)]
    pid = patient["patientAccountId"]

    # Mix of recent completed visits and upcoming scheduled visits
    if i < 12:
        appt_date = REFERENCE_DATE - timedelta(days=random.randint(1, 14))
        status = random.choices(["Completed", "NoShow", "Cancelled"], weights=[0.75, 0.15, 0.10])[0]
    else:
        appt_date = REFERENCE_DATE + timedelta(days=random.randint(0, 14))
        status = "Scheduled"

    # Deliberately vary lead times (0 to 55 days) and ages to exercise
    # both Routine and High no-show risk tiers in Denis's model
    if i % 4 == 0:
        lead_time = random.randint(25, 55)  # higher no-show risk pattern
        age = random.randint(19, 32)
    elif i % 4 == 1:
        lead_time = 0                        # same-day / lower risk pattern
        age = random.randint(55, 74)
    else:
        lead_time = random.randint(2, 21)
        age = random.randint(12, 80)

    # Pure Crystal-style record (matches GetAppointments in demo_response)
    appointments.append({
        "scheduleId": schedule_id,
        "patientId": pid,
        "appointmentDate": appt_date.isoformat(),
        "appointmentTime": random.choice(APPT_TIMES),
        "providerId": random.choice([1, 2]),
        "locationId": 1,
        "firstName": patient["firstName"],
        "lastName": patient["lastName"],
        "status": status
    })

    # Synthetic application-only features (explicitly separated from Crystal schema)
    scheduling_booking_features.append({
        "scheduleId": schedule_id,
        "patientId": pid,
        "appointmentDate": appt_date.isoformat(),
        "status": status,
        "appointment_type": random.choice(APPT_TYPES),
        "schema_origin": "synthetic_application_only",
        # Denis's 10 REQUIRED_FEATURES (exact names & valid ranges)
        "Age": int(age),
        "LeadTime_Days": int(lead_time),
        "Scholarship": int(random.choices([0, 1], weights=[0.88, 0.12])[0]),
        "Hipertension": int(1 if age >= 55 and random.random() < 0.35 else 0),
        "Diabetes": int(1 if age >= 50 and random.random() < 0.18 else 0),
        "Alcoholism": int(random.choices([0, 1], weights=[0.96, 0.04])[0]),
        "Handcap": int(random.choices([0, 1], weights=[0.97, 0.03])[0]),
        "Gender": random.choice(["F", "M"]),
        "Neighbourhood": random.choice(VALID_NEIGHBOURHOODS),
        "Appointment_Weekday": appt_date.strftime("%A")
    })

# ---------------------------------------------------------------------
# 3. Synthetic Insurance Verifications (Matches TEMP_INSURANCE_RECORDS)
# ---------------------------------------------------------------------
PAYERS = ["VSP Choice", "EyeMed Insight", "Spectera", "Davis Vision", "Medicare Medical"]
INS_EXCEPTIONS = [
    "Verification incomplete",
    "Member ID mismatch on payer portal",
    "Annual exam benefit eligibility not confirmed",
    "Secondary medical coverage requires manual confirmation",
    "Prior authorization pending for medical follow-up"
]

insurance_verifications = []
for idx, patient in enumerate(patients):
    pid = patient["patientAccountId"]
    # ~67% Verified, ~33% requiring staff review / exception handling
    if idx % 3 == 0:
        v_status = random.choice(["Needs Review", "Pending", "Expired"])
        exc = INS_EXCEPTIONS[idx % len(INS_EXCEPTIONS)]
        action = "Manually verify eligibility in payer portal prior to appointment."
    else:
        v_status = "Verified"
        exc = None
        action = "No follow-up required."

    insurance_verifications.append({
        # Core keys consumed by Thomas's CapstoneWorkflowGateway & get_dashboard_records
        "patient_id": pid,
        "verification_status": v_status,
        "exception": exc,
        # Synthetic application-only metadata for Front Desk & Owner dashboards
        "patient_name": f"{patient['firstName']} {patient['lastName']}",
        "payer_name": PAYERS[idx % len(PAYERS)],
        "last_checked_date": (REFERENCE_DATE - timedelta(days=random.randint(0, 10))).isoformat(),
        "requires_human_review": v_status != "Verified",
        "recommended_follow_up": action,
        "schema_origin": "synthetic_application_only"
    })

# ---------------------------------------------------------------------
# 4. Synthetic Optical Orders (Crystal-style: GetOfficeFrameOrders &
#    GetOfficeContactLensOrders)
# ---------------------------------------------------------------------
OPTICIANS = ["Optician A", "Optician B", "Optician C"]
LABS = ["Pacific Optical Lab", "Cascade Lens Works", "Northwest RX Lab"]

frame_orders = []
for i in range(25):
    pid = patients[i % len(patients)]["patientAccountId"]
    order_id = str(9001 + i)

    if i < 7:
        # Deliberately include late/overdue InFulfillment orders (dueDate < REFERENCE_DATE)
        order_date = REFERENCE_DATE - timedelta(days=random.randint(18, 28))
        due_date = order_date + timedelta(days=12)  # overdue
        dispense_date = None
        status = "InFulfillment"
    elif i < 12:
        # On-time InFulfillment orders
        order_date = REFERENCE_DATE - timedelta(days=random.randint(2, 7))
        due_date = order_date + timedelta(days=14)
        dispense_date = None
        status = "InFulfillment"
    elif i < 19:
        # Ready for pickup (some waiting > 7 days for pickup follow-up)
        order_date = REFERENCE_DATE - timedelta(days=random.randint(12, 24))
        due_date = order_date + timedelta(days=10)
        dispense_date = None
        status = "ReadyForDispensing"
    else:
        # Dispensed orders
        order_date = REFERENCE_DATE - timedelta(days=random.randint(15, 35))
        due_date = order_date + timedelta(days=12)
        dispense_date = (due_date - timedelta(days=random.randint(0, 2))).isoformat()
        status = "Dispensed"

    frame_orders.append({
        "id": order_id,
        "patientAccountId": pid,
        "status": status,
        "orderDate": order_date.isoformat(),
        "dueDate": due_date.isoformat(),
        "dispenseDate": dispense_date,
        "optician": OPTICIANS[i % len(OPTICIANS)],
        "lab": {"name": LABS[i % len(LABS)]}
    })

contact_lens_orders = []
for i in range(15):
    pid = patients[(i + 5) % len(patients)]["patientAccountId"]
    order_id = str(9501 + i)
    order_date = REFERENCE_DATE - timedelta(days=random.randint(3, 25))
    due_date = order_date + timedelta(days=7)

    if i < 5:
        status, dispense_date = "InFulfillment", None
    elif i < 10:
        status, dispense_date = "ReadyForDispensing", None
    else:
        status = "Dispensed"
        dispense_date = (due_date + timedelta(days=1)).isoformat()

    # Matches Thomas's demo_response shape for GetOfficeContactLensOrders (no optician/lab)
    contact_lens_orders.append({
        "id": order_id,
        "patientAccountId": pid,
        "status": status,
        "orderDate": order_date.isoformat(),
        "dueDate": due_date.isoformat(),
        "dispenseDate": dispense_date
    })

# ---------------------------------------------------------------------
# 5. Save Artifacts & Schema Manifest to Shared Unity Catalog Volume
# ---------------------------------------------------------------------
artifacts = {
    "patients.json": patients,
    "appointments.json": appointments,
    "scheduling_booking_features.json": scheduling_booking_features,
    "insurance_verifications.json": insurance_verifications,
    "frame_orders.json": frame_orders,
    "contact_lens_orders.json": contact_lens_orders,
}

for filename, payload in artifacts.items():
    path = f"{SYNTHETIC_DATA_DIR}/{filename}"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

manifest = {
    "dataset_version": "eye_style_synthetic_mvp_v1",
    "generated_reference_date": REFERENCE_DATE.isoformat(),
    "random_seed": SEED,
    "contains_phi": False,
    "live_crystal_data": False,
    "files": {
        "patients.json": {
            "count": len(patients),
            "crystal_operation": "GetPatientById",
            "schema_origin": "crystal_style_synthetic"
        },
        "appointments.json": {
            "count": len(appointments),
            "crystal_operation": "GetAppointments",
            "schema_origin": "crystal_style_synthetic"
        },
        "scheduling_booking_features.json": {
            "count": len(scheduling_booking_features),
            "crystal_operation": "GetNoShowRisk (input features)",
            "schema_origin": "synthetic_application_only",
            "note": "Contains Denis's 10 booking-time REQUIRED_FEATURES; not confirmed Crystal PM fields."
        },
        "insurance_verifications.json": {
            "count": len(insurance_verifications),
            "crystal_operation": "GetVerificationStatus",
            "schema_origin": "synthetic_application_only",
            "note": "Replaces TEMP_INSURANCE_RECORDS; preserves patient_id, verification_status, exception."
        },
        "frame_orders.json": {
            "count": len(frame_orders),
            "crystal_operation": "GetOfficeFrameOrders",
            "schema_origin": "crystal_style_synthetic"
        },
        "contact_lens_orders.json": {
            "count": len(contact_lens_orders),
            "crystal_operation": "GetOfficeContactLensOrders",
            "schema_origin": "crystal_style_synthetic"
        }
    }
}

manifest_path = f"{SYNTHETIC_DATA_DIR}/synthetic_data_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

# ---------------------------------------------------------------------
# 6. Verification & Compatibility Smoke Test
# ---------------------------------------------------------------------
print("=== SYNTHETIC CRYSTAL-STYLE MVP DATASET GENERATED ===")
for fname, payload in artifacts.items():
    print(f"- {fname}: {len(payload)} records")

# Verify overdue optical orders and insurance exceptions exist for demo
late_frames = [
    o for o in frame_orders
    if o["status"] == "InFulfillment" and o["dueDate"] < REFERENCE_DATE.isoformat()
]
unresolved_ins = [
    r for r in insurance_verifications if r["verification_status"] != "Verified"
]
print(f"\nDemo edge cases verified:")
print(f"- Overdue InFulfillment frame orders: {len(late_frames)}")
print(f"- Unresolved insurance verifications: {len(unresolved_ins)}")

# Smoke-test Denis's no-show model scoring
if "predict_no_show" in globals():
    scored = []
    for feat_row in scheduling_booking_features:
        if feat_row["status"] == "Scheduled":
            pred = predict_no_show(feat_row)
            flagged = bool(pred["follow_up_flag"])
            scored.append({
                "scheduleId": feat_row["scheduleId"],
                "patientId": feat_row["patientId"],
                "appointmentDate": feat_row["appointmentDate"],
                "status": feat_row["status"],
                "data_mode": "demo",
                "model_version": pred["model_version"],
                "no_show_score": float(pred["no_show_score"]),
                "threshold": float(pred["threshold"]),
                "follow_up_flag": flagged,
                "priority": "High" if flagged else "Routine",
                "recommended_action": (
                    "Review appointment and consider staff follow-up."
                    if flagged else
                    "Continue normal scheduling workflow."
                ),
                "requires_human_review": flagged,
                "automatic_contact": False,
                "prediction_point": pred["prediction_point"],
            })
    high_risk = sum(1 for r in scored if r["follow_up_flag"])
    print(f"- Denis MLP smoke test PASSED: scored {len(scored)} scheduled appointments "
          f"({high_risk} flagged High priority, {len(scored) - high_risk} Routine)")
    with open(f"{SYNTHETIC_DATA_DIR}/scheduled_no_show_scores.json", "w", encoding="utf-8") as f:
        json.dump(scored, f, indent=2)
else:
    print("- Note: Run Denis's inference cell first to also export scheduled_no_show_scores.json.")

In [ ]:
# Import the extracted synthetic bundle loader.
from src.data.synthetic_loader import load_synthetic_mvp_data
